# 04 · Tools and structured outputs

**Session 01 · Building AI applications for research and data · notebook 4 of 6**

## Objective

Notebook 03 ended on a limit of search. Even when the right documents come back, the passages may not hold the figures an answer needs: for the main question, the four passages held two of the four numbers. For numbers, the application can skip the search and ask a tool instead, which looks the figure up in the data itself. This notebook gives the model such a tool, watches it use the tool, makes its answer a structured output that other software can read, and then offers the same tool to other programs.

The application so far keeps documents in a store, searches the store, and hands the model the passages it finds (notebook 03). That stays, for definitions and context. This notebook adds a second source of evidence, a tool for exact numbers, and a second kind of answer.

Four terms, in plain words:

- A **tool** is a function that the application can run for the model. The model cannot run it itself: it asks, the application runs it, and the result goes back to the model.
- A **structured output** is an answer returned in a fixed shape, with named fields, instead of free text, so that another program can read it without guessing.
- A **schema** is the description of that shape: the named fields and their types ("year: a whole number") that the answer must fill in.
- **MCP**, the Model Context Protocol, is a standard plug for tools. A tool offered through it can be used by any program that speaks the standard, such as a chatbot or a desktop assistant.

## The application after this notebook

![Diagram: the question goes to a search of the store, which returns passages for the prompt; the prompt goes to the model. New in this notebook: the model can call the tool get_indicator with an indicator, a country and optional start and end years; the data it returns goes back into the prompt; the model answers with text or a structured output, a filled-in form that other software can read. Separately, the same tool is served over MCP to any MCP client, such as this notebook, a chatbot or a desktop assistant.]()

The light boxes are what notebooks 01 to 03 built. The dark boxes are what this notebook adds: the tool call and the data it returns, the structured output, and the same tool served over MCP. Passages are for definitions and context. Numbers come from a tool over the data.

## What you will see

Five short scenes:

1. The four figures that the main question needs: what the search of notebook 03 found, and what the tool returns.
2. The tool as the model sees it, and an animation of the model asking for it, step by step.
3. A year that has no data: the tool says so, and the model passes it on.
4. A structured output: a program that misreads a sentence, then the same answer in a blank form that the model fills in, and what happens when a box is damaged.
5. The same tool offered over MCP and used by an agent, and a call to the program's public server.

## How to run it

Run the cells in order. The model's answers are recorded ones, labelled `[recorded output: ...]`, as in notebook 01. The tool results, tables and checks are computed when you run the cell. The last cell reaches a public server and needs the internet.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "main"   # the branch the Colab setup clones
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


## 1. Search and tool, for the same four figures

Section 6 of notebook 03 found that the main question needs four numbers: internet use and GDP per capita, for Kenya and Rwanda, in 2024. The cell rebuilds the store of notebook 03 in three lines, runs the same search for the four closest passages, and checks for each number whether it is in them. Beside it, the tool's answer for the same figure.

In [2]:
docs = w.load_documents()
store = w.build_store(w.split_sections(docs))   # the store of notebooks 02 and 03

needed = next(item for item in w.load_questions() if item["id"] == "q1")["expected_numbers"]
passages = store.similarity_search(w.MAIN_QUESTION, k=4)
text = " ".join(p.page_content for p in passages)

countries = {"KEN": "Kenya", "RWA": "Rwanda"}
figures = {"IT.NET.USER.ZS": ("internet use, % of population", "{:.1f}"), "NY.GDP.PCAP.CD": ("GDP per capita, US dollars", "{:,.0f}")}

rows = []
for n in needed:
    name, fmt = figures[n["indicator"]]
    result = w.indicator_lookup(n["indicator"], countries[n["country"]], start=n["year"], end=n["year"])
    rows.append((countries[n["country"]], name, n["year"],
                 "hit" if w.numbers_present(text, [n])[0] == 1 else "miss",
                 fmt.format(w.latest(result)["value"])))
w.show_table(["country", "figure", "year", "in the 4 passages", "from the tool"], rows, source="The search is the one of notebook 03, section 1.")

country,figure,year,in the 4 passages,from the tool
Kenya,"internet use, % of population",2024,hit,35.0
Rwanda,"internet use, % of population",2024,hit,31.7
Kenya,"GDP per capita, US dollars",2024,miss,"2,133"
Rwanda,"GDP per capita, US dollars",2024,miss,"1,060"


The search found two of the four. The tool returned all four, each with its indicator, year and source. A tool does not guess which passage might hold a figure: the application asks for the figure by name.

The model's own memory is a third way to get a number, and the weakest. The cell asks the model for Rwanda's GDP per capita in 2023, then asks the tool.

In [3]:
w.use("nb04_from_memory")
w.show(model.invoke("What was Rwanda's GDP per capita in 2023?"))
print()
r = w.indicator_lookup("NY.GDP.PCAP.CD", "Rwanda", start=2023, end=2023)
w.show_table(["year", "value"], [(o["year"], f"{o['value']:,.1f}") for o in r["observations"]], source=f"{r['name']} · {r['source']}")

`[recorded output: nb04_from_memory]`

Rwanda's GDP per capita in 2023 was approximately 1,000 US dollars (current prices), according to World Bank data. Please verify against the latest World Development Indicators, as figures are revised.

year,value
2023,"1,061.1"


The answer from memory is close and hedged, and nothing tells a reader how close. The tool's answer is exact and carries its source. The application can now get an exact number, with its source, instead of hoping that a passage holds it.

## 2. The tool, as the model sees it

`get_indicator` is a Python function decorated with LangChain's `@tool`. Its docstring becomes the description the model reads, and its type hints become the form the model fills in: one field for each parameter. The cell shows both, and the five indicators the data holds.

In [4]:
print("name:       ", w.get_indicator.name)
print("description:", w.get_indicator.description)
print()

meaning = {"indicator": "which series, as an id such as IT.NET.USER.ZS", "country": "a name or a three-letter code",
           "start": "the first year wanted", "end": "the last year wanted"}
kinds = {"string": "text", "integer": "whole number"}
fields = []
for field, spec in w.get_indicator.args.items():
    kind = spec.get("type") or " or ".join(a["type"] for a in spec["anyOf"] if a["type"] != "null")
    fields.append((field, kinds.get(kind, kind), "optional" if "default" in spec else "required", meaning[field]))
w.show_table(["field", "type", "", "meaning"], fields, source="The form the model fills in, from the tool's type hints.")
print()
w.show_table(["indicator id", "name"], [(i["id"], i["name"]) for i in w.list_indicators()], source="The series the tool can look up.")

name:        get_indicator
description: Observations of a World Development Indicators series for one country. Use it for
    any statistic; do not quote figures from memory. `indicator` is an id such as
    IT.NET.USER.ZS or NY.GDP.PCAP.CD; `country` is a name or ISO3 code; `start` and `end`
    bound the years.



field,type,,meaning
indicator,text,required,"which series, as an id such as IT.NET.USER.ZS"
country,text,required,a name or a three-letter code
start,whole number,optional,the first year wanted
end,whole number,optional,the last year wanted


indicator id,name
NY.GDP.PCAP.CD,GDP per capita (current US$)
IT.NET.USER.ZS,Individuals using the Internet (% of population)
NY.GDP.MKTP.KD.ZG,GDP growth (annual %)
SP.POP.TOTL,"Population, total"
IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people)


## 3. The model asks, the application runs the tool

A model does not run tools. It replies with a request that names the tool and fills in the form, the application runs the tool, and the result goes back to the model, which then answers. The animation shows the exchange.

![Animation: the application sends the model the question and the list of tools; the model replies with a request for get_indicator with GDP per capita, Rwanda and 2023; the application runs the tool, which returns 1,061 with its source; the application hands that back and the model answers that it was 1,061 US dollars in 2023.]()

`bind_tools` tells the model which tools exist. The cell does the exchange once, written out, and labels the three steps. Notebook 05 lets LangGraph run this loop.

In [5]:
import json
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

INSTRUCTIONS = """You are a research assistant for development data.
Use get_indicator for every statistic; do not quote figures from memory. Name the indicator and the source with each figure.
If the tool returns no observation for the year asked, say so and give the latest available year."""

model_with_tools = model.bind_tools([w.get_indicator])
messages = [SystemMessage(content=INSTRUCTIONS), HumanMessage(content="What was Rwanda's GDP per capita in 2023?")]

w.use("nb04_tool_call")
print("1. The model asks for the tool:")
reply = model_with_tools.invoke(messages)
w.show(reply)
for call in reply.tool_calls:
    result = w.get_indicator.invoke(call["args"])
    print()
    print("2. The application runs it, and the tool returns:")
    w.show_table(["year", "value"], [(o["year"], f"{o['value']:,.1f}") for o in result["observations"]], source=f"{result['name']} · {result['source']}")
    messages += [reply, ToolMessage(content=json.dumps(result), tool_call_id=call["id"])]
print()
print("3. The model writes its answer from the result:")
final = model_with_tools.invoke(messages)
w.show(final)

1. The model asks for the tool:


`[recorded output: nb04_tool_call]`

`-> tool call get_indicator({"indicator": "NY.GDP.PCAP.CD", "country": "Rwanda", "start": 2020, "end": 2024})`




2. The application runs it, and the tool returns:


year,value
2020,802.6
2021,842.6
2022,"1,007.1"
2023,"1,061.1"
2024,"1,059.9"



3. The model writes its answer from the result:


`[recorded output: nb04_tool_call]`

Rwanda's GDP per capita in 2023 was 1,061 US dollars in current prices (World Development Indicators, NY.GDP.PCAP.CD). It rose from 803 dollars in 2020 to 1,007 in 2022 and 1,061 in 2023, and was 1,060 in 2024.

The application can now use a tool: the model asks, the application runs it, and the result goes back to the model.

## 4. Data that does not exist

The planted problem: a year with no observation. The tool returns an empty list, and the instructions say what to do with it. The cell prints what the tool returned, then the model's answer.

In [6]:
messages = [SystemMessage(content=INSTRUCTIONS), HumanMessage(content="What share of Kenya's population used the internet in 2025?")]
w.use("nb04_missing_year")
reply = model_with_tools.invoke(messages)
for call in reply.tool_calls:
    result = w.get_indicator.invoke(call["args"])
    print("tool result for 2025:", result["observations"])
    messages += [reply, ToolMessage(content=json.dumps(result), tool_call_id=call["id"])]
print()
w.show(model_with_tools.invoke(messages))

tool result for 2025: []



`[recorded output: nb04_missing_year]`

No observation is available for 2025. The series IT.NET.USER.ZS (Individuals using the Internet, % of population) has no 2025 value for Kenya in the data returned. The latest available year is 2024, when the value was 35.0 percent.

## 5. Structured output: an answer in a fixed shape

So far the model's answers are sentences, which suits a person. Suppose instead that a dashboard has a box labelled "GDP per capita, Rwanda" and must fill it from the model's answer. The cell does what a simple program might do: it takes the first number in the sentence from section 3.

In [7]:
import re

first = re.findall(r"\d[\d,]*\.?\d*", final.content)[0]
print("The answer:")
w.show(final.content)
print("\nThe dashboard's box: ", first)

The answer:


Rwanda's GDP per capita in 2023 was 1,061 US dollars in current prices (World Development Indicators, NY.GDP.PCAP.CD). It rose from 803 dollars in 2020 to 1,007 in 2022 and 1,061 in 2023, and was 1,060 in 2024.


The dashboard's box:  2023


The box gets 2023, which is the year and not a figure. The sentence holds three figures for different years, and the words around them change from one answer to the next, so a program reading a sentence has to guess which one is wanted.

The fix is to stop asking for a sentence. We hand the model a **blank form** with a named box for each thing we need: a country, a year and a value for internet use, a year and a value for GDP per capita, a comparison, the sources. The model's job is to fill in the boxes, and the program reads each box by name. The blank form is called a **schema**, and an answer that follows one is a **structured output**. In code the form is a class written with a library called Pydantic, and `with_structured_output` sends it to the model with the request. The next cell gives the model the tool's data, asks it to fill in the form, and draws the blank form beside the filled one.

In [8]:
data = [w.indicator_lookup(i, c, start=2024) for i in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"] for c in ["Kenya", "Rwanda"]]
structured = model.with_structured_output(w.ComparisonAnswer)

w.use("nb04_structured")
parsed = structured.invoke([
    SystemMessage(content="Fill the schema from the data you are given. Use the values exactly as given."),
    HumanMessage(content="DATA:\n" + json.dumps(data) + "\n\nQUESTION: " + w.MAIN_QUESTION),
])
w.show_form(w.ComparisonAnswer, parsed)

Now the dashboard does not guess. It reads the box it needs by its name, and gets the figure and its year.

In [9]:
box = next(cs for cs in parsed.countries if cs.country == "RWA").gdp_per_capita_latest
print("GDP per capita, Rwanda:", f"{box.value:,.0f} US dollars, in {box.year}")

GDP per capita, Rwanda: 1,060 US dollars, in 2024


A form also has rules, and the program checks the answer against them before using it. A box that should hold a year cannot hold words. The cell damages the answer by putting words in a year box, and shows the message the program would receive, which names the exact box. Without the check, a wrong value would pass through with no error.

In [10]:
from pydantic import ValidationError

damaged = parsed.model_dump_json().replace('"year":2024', '"year":"twenty twenty-four"', 1)
try:
    w.ComparisonAnswer.model_validate_json(damaged)
except ValidationError as e:
    box = " > ".join(str(part) for part in e.errors()[0]["loc"])
    print("box with the problem:", box)
    print("what is wrong:       ", e.errors()[0]["msg"])

box with the problem: countries > 0 > internet_latest > year
what is wrong:        Input should be a valid integer, unable to parse string as an integer


The application can now return an answer that other software reads box by box, and reject one that breaks the form.

## 6. The same tool over MCP

MCP is the standard plug: a tool written once as a server can be used by any program that speaks the standard, whether a chatbot, a desktop assistant or an agent. `indicator_server.py` wraps `get_indicator` in a FastMCP server in a dozen lines. The FastMCP client below starts it, lists its tools and calls one.

In [11]:
from fastmcp import Client

async with Client(w.mcp_server_transport()) as client:
    for t in await client.list_tools():
        print(f"{t.name:16s} {t.description[:90]}")
    result = await client.call_tool("get_indicator", {"indicator": "IT.NET.USER.ZS", "country": "Rwanda", "start": 2022})
    data = result.data
    print()
    w.show_table(["year", "value"], [(o["year"], f"{o['value']:,.1f}") for o in data["observations"]], source=f"{data['name']} · {data['source']}")

list_indicators  The indicators this server holds, with their ids and names.
get_indicator    Observations of a World Development Indicators series for one country. `indicator`
is an i



year,value
2022,28.0
2023,28.8
2024,31.7


A LangChain tool can wrap the MCP call, so an agent uses the server like any other tool. The cell defines one and runs an agent over it. An agent is a model that chooses its own tool calls; notebook 05 is about agents.

In [12]:
from langchain_core.tools import tool
from langchain.agents import create_agent


@tool("get_indicator")
async def get_indicator_over_mcp(indicator: str, country: str, start: int | None = None, end: int | None = None) -> dict:
    """Observations of a World Development Indicators series for one country, from the indicator MCP server."""
    async with Client(w.mcp_server_transport()) as client:
        args = {k: v for k, v in dict(indicator=indicator, country=country, start=start, end=end).items() if v is not None}
        return (await client.call_tool("get_indicator", args)).data


agent = create_agent(model, [get_indicator_over_mcp], system_prompt=INSTRUCTIONS)
w.use("nb04_mcp")
result = await agent.ainvoke({"messages": [HumanMessage(content="How has internet use in Rwanda changed since 2022?")]})
w.print_trace(result)
print()
w.show(w.final_text(result))

step 1: get_indicator({"indicator": "IT.NET.USER.ZS", "country": "Rwanda", "start": 2022, "end": 2024})
   -> {"indicator": "IT.NET.USER.ZS", "country": "RWA", "name": "Individuals using the Internet …



Rwanda's share of individuals using the internet was 28.0 percent in 2022, 28.8 percent in 2023, and 31.7 percent in 2024 (World Development Indicators, IT.NET.USER.ZS, through the indicator MCP server).

The application can now offer its tool to any program that speaks MCP.

The program's [Data360 MCP server](https://github.com/worldbank/data360-mcp) offers search, metadata and data tools over the World Bank's data platform in the same way, at a public address. The cell lists its tools and runs one search. It reaches a live server, so the list can differ from run to run, and if the call fails the cell says so and the notebook carries on.

In [13]:
try:
    async with Client(w.DATA360_MCP_URL) as client:
        tools = await client.list_tools()
        print(len(tools), "tools, for example:")
        for t in tools[:5]:
            print("  ", t.name)
        found = await client.call_tool("data360_search_indicators", {"query": "individuals using the internet", "limit": 3})
        for ind in found.data["indicators"]:
            print(f"  {ind['idno']:28s} {ind['name']}")
except Exception as e:
    print("Data360 MCP call skipped:", type(e).__name__)

16 tools, for example:
   data360_search_indicators
   data360_search_datasets
   data360_get_metadata
   data360_get_data
   data360_get_disaggregation


  WEF_GCIHH_NETUSERPCT         Individuals using Internet, %
  WB_WDI_IT_NET_USER_ZS        Individuals using the Internet (% of population)
  WEF_TTDI_NETUSERPCT          Individuals using Internet, % of pop


## Key points

- Numbers come from a tool over authoritative data, and the model's memory is for language. In this notebook the search found two of the four figures the main question needs, and the tool returned all four.
- A tool is a function with a description the model can read and a form the model fills in. The model does not run it: the application does, and hands the result back.
- Absence of data is a tool result that the application can test for, instead of a gap the model fills with a guess.
- A structured output is an answer in a fixed shape: the model fills in named fields instead of writing free text, which a program would have to guess at. The schema is the contract: checked on the way in, readable by other software on the way out.
- MCP is the standard plug: the same tool, written once as a server, is available to any program that speaks it.

## Exercise

1. In section 1, change `k=4` to `k=8` and run the cell. How many of the four figures does the search find now, and what does the extra search cost in passages? Does the tool's column change?
2. The data holds population too, series `SP.POP.TOTL`. Run `w.indicator_lookup("SP.POP.TOTL", "Rwanda", start=2024, end=2024)` and `w.indicator_lookup("IT.NET.USER.ZS", "Rwanda", start=2024, end=2024)`, and work out how many people in Rwanda used the internet in 2024. An answer like that needs two tool results and a calculation: should the model do the arithmetic, or should the application?
3. Run `w.indicator_lookup("IT.NET.USER.ZS", "Kenya", start=2024, end=2025)`. The tool returns the year that exists and says nothing about the one that does not. Why does the instruction "if the tool returns no observation for the year asked, say so" matter for a result like this?
4. In section 5, damage a different field of the answer, for example replace a value with the word "high", and read the error. Which field does it name?

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.

**Next:** [05 · Workflows and agents](https://colab.research.google.com/github/worldbank/ai4data/blob/main/training/session-01-building-ai-applications/notebooks/05-workflows-and-agents.ipynb)